In [19]:
# %% [markdown]
# # Portfolio Analyzer: yfinance API Integration & Industry Tracking
# This script reads a Quicken holdings export, pings yfinance for live
# asset classification and sector data, and calculates precise portfolio exposure.

# %%
import io
import re
import time
import pandas as pd
import numpy as np
import yfinance as yf

CSV_FILENAME = "data/Chris's Finances - Investing - Portfolio Value - By Managed Account - 2026-09-24.csv"

# %%
def load_and_parse_holdings(filepath):
    """Parses the Quicken 'Portfolio Value' export into a clean Account/Symbol DataFrame."""
    try:
        with open(filepath, 'rb') as f:
            raw_bytes = f.read()
    except FileNotFoundError:
        print(f"Error: Could not find '{filepath}'.")
        return None

    text = raw_bytes.decode('utf-8-sig', errors='replace').replace('\ufeff', '')
    lines = text.splitlines(keepends=True)

    header_idx = next((i for i, line in enumerate(lines) if "symbol" in line.lower() and "market value" in line.lower()), None)
    if header_idx is None:
        print("Error: Could not find table header row.")
        return None

    df = pd.read_csv(io.StringIO("".join(lines[header_idx:])), sep=',', on_bad_lines='skip')

    col_names = list(df.columns)
    col_names[0] = 'Name'
    df.columns = col_names

    if 'Market Value' in df.columns:
        df['Market Value'] = df['Market Value'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip()
        df['Market Value'] = pd.to_numeric(df['Market Value'], errors='coerce')

    parsed_data = []
    current_account = "Unknown Account"

    for _, row in df.iterrows():
        raw_name = str(row['Name'])
        if pd.isna(row['Name']) or raw_name.strip() == '':
            continue

        # Top-level accounts in Quicken have NO leading spaces. Holdings are indented.
        if not raw_name.startswith(' ') and not raw_name.startswith('\t'):
            current_account = raw_name.strip()
        else:
            # It's an indented holding, even if the symbol is completely blank
            name_val = raw_name.strip()

            # Handle missing symbols for Munis/Cash without creating a new account
            if pd.isna(row['Symbol']):
                symbol = "CASH" if "cash" in name_val.lower() else "UNLISTED_ASSET"
            else:
                symbol = str(row['Symbol']).strip().upper()

            market_value = row['Market Value'] if not pd.isna(row['Market Value']) else 0.0

            parsed_data.append({
                'Account': current_account,
                'Asset_Name': name_val,
                'Symbol': symbol,
                'Market_Value': market_value
            })

    return pd.DataFrame(parsed_data)

# %%

def fetch_yfinance_metadata(df):
    """
    Pings yfinance for live metadata. Relies on Quicken names for accuracy,
    detects Min Volatility / Muni strategies, and skips unlisted assets.
    """
    print("Pinging yfinance for live asset metadata (with rate-limit protection)...")

    # 1. Remap outdated legacy tickers to their current active symbols
    df['Symbol'] = df['Symbol'].replace({
        'ERJ': 'EMBJ',
        'ABC': 'COR',
        '*BDANOW': 'CASH',
        'USD=': 'CASH'
    })

    # 2. Bypass yfinance for known database glitches and unlisted assets
    MANUAL_OVERRIDES = {
        'META': {'Asset_Class': 'Individual Security', 'Sector_Category': 'Technology - Communication Services'},
        'PEP': {'Asset_Class': 'Individual Security', 'Sector_Category': 'Consumer Defensive - Beverages'},
        'VSCO': {'Asset_Class': 'Individual Security', 'Sector_Category': 'Consumer Cyclical - Specialty Retail'},
        'IAC': {'Asset_Class': 'Individual Security', 'Sector_Category': 'Technology - Internet Content & Information'},
        'BK': {'Asset_Class': 'Individual Security', 'Sector_Category': 'Financial Services - Asset Management'},
        'UNLISTED_ASSET': {'Asset_Class': 'Municipal Bond', 'Sector_Category': 'Individual Bond / Unlisted'}
    }

    unique_symbols = [sym for sym in df['Symbol'].unique() if sym != "CASH" and str(sym) != "NAN"]
    metadata_dict = {}

    for sym in unique_symbols:
        # Check the bypass dictionary FIRST
        if sym in MANUAL_OVERRIDES:
            metadata_dict[sym] = MANUAL_OVERRIDES[sym]
            continue

        # Also catch bonds/CUSIPs before pinging
        if (len(sym) == 9 and sym.isalnum()) or sym.startswith('M1') or sym.endswith('-'):
            metadata_dict[sym] = {'Asset_Class': 'Municipal Bond', 'Sector_Category': 'Individual Bond / Unlisted'}
            continue

        max_retries = 3
        for attempt in range(max_retries):
            try:
                # Force known glitching tickers to individual equity without pinging if needed
                if sym in ['VSCO', 'IAC', 'BK']:
                    metadata_dict[sym] = {'Asset_Class': 'Individual Security', 'Sector_Category': 'Consumer / Financial (Manual)'}
                    break

                info = yf.Ticker(sym).info

                if not info or 'quoteType' not in info:
                    raise ValueError(f"Empty info returned for {sym}")

                quote_type = info.get('quoteType', 'UNKNOWN')

                if quote_type == 'EQUITY':
                    asset_class = 'Individual Security'
                    category = f"{info.get('sector', 'Unknown Sector')} - {info.get('industry', 'Unknown Industry')}"
                elif quote_type in ['ETF', 'MUTUALFUND']:
                    asset_class = 'ETF / Index'
                    category = info.get('category', 'Unknown Fund Category')
                else:
                    asset_class = 'Other'
                    category = quote_type

                metadata_dict[sym] = {'Asset_Class': asset_class, 'Sector_Category': category}
                break

            except Exception:
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                else:
                    metadata_dict[sym] = {'Asset_Class': 'Unknown', 'Sector_Category': 'API Error / Not Found'}

        time.sleep(0.3)

    # 3. Map fetched data back to the dataframe
    df['Asset_Class'] = df['Symbol'].apply(lambda x: 'Cash' if x == 'CASH' else metadata_dict.get(x, {}).get('Asset_Class', 'Unknown'))
    df['Sector_Category'] = df['Symbol'].apply(lambda x: 'Cash Equivalent' if x == 'CASH' else metadata_dict.get(x, {}).get('Sector_Category', 'Unknown'))

    # 4. Keyword Override: Force correct mapping based on the actual Quicken Name
    # 4. Keyword Override: Force correct mapping based on the actual Quicken Name
    def override_category(row):
        name = str(row['Asset_Name']).lower()
        ac = str(row['Asset_Class'])
        cat = str(row['Sector_Category'])

        # Complex Equity Strategies
        if 'emerging mkt' in name and ('low vol' in name or 'low volatiliy' in name):
            ac = 'ETF / Index'
            cat = 'Emerging Markets Low Volatility'
        elif 'min vol' in name or 'low volatility' in name:
            ac = 'ETF / Index'
            cat = 'Low Volatility Equity'

        # Specialized Fixed Income
        elif 'tax free' in name or 'tax sensitive' in name or 'muni' in name:
            ac = 'Municipal Bond'
            cat = 'Municipal Fixed Income'
        elif 'treasury' in name:
            ac = 'ETF / Index'
            cat = 'Treasury Fixed Income'
        elif 'inflat-prot' in name or 'tips' in name:
            ac = 'ETF / Index'
            cat = 'Inflation-Protected Bonds (TIPS)'

        return pd.Series([ac, cat])

    df[['Asset_Class', 'Sector_Category']] = df.apply(override_category, axis=1)

    return df

# %%
def analyze_portfolio_with_sectors(df):
    """Outputs the portfolio breakdown including Sector/Industry analysis."""
    if df is None or df.empty:
        return

    accounts = df['Account'].unique()

    print("\n" + "="*70)
    print("PORTFOLIO HOLDINGS & SECTOR EXPOSURE ANALYSIS")
    print("="*70)

    for acc in accounts:
        acc_df = df[df['Account'] == acc].copy()
        total_value = acc_df['Market_Value'].sum()

        if total_value == 0:
            continue

        acc_df['Weight_%'] = (acc_df['Market_Value'] / total_value) * 100

        # Determine True Underlying Asset Allocation (Macro Split)
        def classify_macro(row):
            cat = str(row['Sector_Category']).lower()
            name = str(row['Asset_Name']).lower()
            ac = str(row['Asset_Class']).lower()

            # If it says bond, muni, cash, or fixed income anywhere, it's safe money
            if 'muni' in cat or 'bond' in cat or 'fixed income' in cat or 'cash' in ac or 'muni' in name:
                return 'Fixed Income / Cash'
            # Otherwise, whether it's an individual stock or an equity ETF, it's equity
            return 'Equity'

        acc_df['Macro_Class'] = acc_df.apply(classify_macro, axis=1)

        # Calculate True Portfolio Ratios
        macro_mix = acc_df.groupby('Macro_Class')['Weight_%'].sum().to_dict()
        true_equity_weight = macro_mix.get('Equity', 0)
        true_fixed_weight = macro_mix.get('Fixed Income / Cash', 0)

        # Determine smarter benchmarks based on TRUE Equity vs Fixed Income ratio
        if true_equity_weight > 90:
            benchmark = "SPY (S&P 500) or ACWI (Global Equity) - 100% Equity"
        elif true_equity_weight > 70:
            benchmark = "AOA (80/20 Aggressive Growth Allocation)"
        elif true_equity_weight > 50:
            benchmark = "AOR (60/40 Core Growth Allocation)"
        elif true_fixed_weight > 80:
            benchmark = "MUB (Muni Bonds) or AGG (Core US Bond) - Fixed Income"
        else:
            benchmark = "AOM (40/60 Conservative Allocation)"

        print(f"\nACCOUNT: {acc}")
        print(f"Total Value:        ${total_value:,.2f}")
        # Print the true risk profile instead of just structural wrappers
        print(f"Macro Allocation:   {true_equity_weight:.1f}% Total Equity | {true_fixed_weight:.1f}% Fixed Income & Cash")
        print(f"Target Benchmark:   {benchmark}")
        print("-" * 50)

        # Display Top Holdings
        top_holdings = acc_df.sort_values(by='Market_Value', ascending=False).head(3)
        print("Top Holdings:")
        for _, row in top_holdings.iterrows():
            asset_name = str(row['Asset_Name'])[:25]
            sector_cat = str(row['Sector_Category']).replace('Unknown Fund Category', 'Fund (Inferred via Name)')[:35]
            print(f"  - {row['Symbol']:<6} | {asset_name:<25} | {row['Weight_%']:>4.1f}% | {sector_cat}")

        # Display the aggregate Sector / Strategy exposure
        print("\nPrimary Sector / Fund Strategy Exposure:")
        sector_exposure = acc_df.groupby('Sector_Category')['Weight_%'].sum().sort_values(ascending=False).head(5)
        for sector, weight in sector_exposure.items():
            if weight > 1.0:
                print(f"  > {sector[:40]:<40} : {weight:>4.1f}%")


# %%
if __name__ == "__main__":
    raw_df = load_and_parse_holdings(CSV_FILENAME)
    if raw_df is not None:
        enriched_df = fetch_yfinance_metadata(raw_df)
        analyze_portfolio_with_sectors(enriched_df)

Pinging yfinance for live asset metadata (with rate-limit protection)...


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ERJ"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ERJ"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ERJ"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ABC"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ABC"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ABC"}}}


KeyboardInterrupt: 

In [ ]:
# %%
from IPython.display import display, HTML
import pandas as pd

def generate_tilt_and_interactive_view(df):
    if df is None or df.empty:
        print("No data available to display.")
        return

    view_df = df.copy()

    # 1. Add the Macro Category (Fixed Income vs Equity)
    def classify_macro(row):
        cat = str(row.get('Sector_Category', '')).lower()
        name = str(row.get('Asset_Name', '')).lower()
        ac = str(row.get('Asset_Class', '')).lower()

        if 'muni' in cat or 'bond' in cat or 'fixed income' in cat or 'cash' in ac or 'muni' in name or 'treasury' in cat or 'tips' in cat:
            return 'Fixed Income & Cash'
        return 'Equity'

    view_df['Macro_Class'] = view_df.apply(classify_macro, axis=1)

    # Calculate account weights
    view_df['Account_Total'] = view_df.groupby('Account')['Market_Value'].transform('sum')
    view_df['Weight_%_Num'] = (view_df['Market_Value'] / view_df['Account_Total']) * 100

    # Format strings for the display table
    view_df['Weight_%'] = view_df['Weight_%_Num'].map('{:.1f}%'.format)
    view_df['Market_Value_Str'] = view_df['Market_Value'].apply(lambda x: f"${x:,.2f}")

    # Sort for the hierarchical table
    view_df = view_df.sort_values(by=['Account', 'Macro_Class', 'Asset_Class', 'Sector_Category', 'Market_Value'],
                                  ascending=[True, False, True, True, False])

    # 2. Reorder columns and generate the nested Jupyter Table
    hierarchical_df = view_df.set_index(['Account', 'Macro_Class', 'Asset_Class', 'Sector_Category', 'Symbol'])
    display_df = hierarchical_df[['Asset_Name', 'Weight_%', 'Market_Value_Str']]
    display_df = display_df.rename(columns={'Market_Value_Str': 'Market_Value'})

    print("\n" + "="*80)
    print("INTERACTIVE PORTFOLIO HIERARCHY")
    print("="*80)
    display(display_df)

    # 3. Portfolio Tilt & Volatility Analysis
    print("\n" + "="*80)
    print("OVERALL PORTFOLIO VOLATILITY & TILT ANALYSIS")
    print("="*80)

    # Define Defensive vs Cyclical/Growth Sectors for Individual Stocks
    defensive_sectors = ['Healthcare', 'Utilities', 'Consumer Defensive', 'Consumer Staples']

    for acc in view_df['Account'].unique():
        acc_df = view_df[view_df['Account'] == acc]
        total_eq = acc_df[acc_df['Macro_Class'] == 'Equity']['Market_Value'].sum()

        if total_eq == 0:
            continue

        acc_df['Eq_Weight'] = (acc_df['Market_Value'] / total_eq) * 100

        # Calculate Targeted Low-Vol ETF Exposure
        low_vol_etfs = acc_df[(acc_df['Asset_Class'] == 'ETF / Index') &
                              (acc_df['Sector_Category'].str.contains('Low Volatility|Min Vol', case=False, na=False))]['Eq_Weight'].sum()

        # Calculate Defensive Individual Stock Exposure
        defensive_stocks = acc_df[(acc_df['Asset_Class'] == 'Individual Security') &
                                  (acc_df['Sector_Category'].str.contains('|'.join(defensive_sectors), case=False, na=False))]['Eq_Weight'].sum()

        # Calculate Aggressive/Growth Tech Exposure (High Beta)
        tech_stocks = acc_df[(acc_df['Asset_Class'] == 'Individual Security') &
                             (acc_df['Sector_Category'].str.contains('Technology|Communication', case=False, na=False))]['Eq_Weight'].sum()

        total_defensive_tilt = low_vol_etfs + defensive_stocks

        print(f"\nAccount: {acc} (Equity Portion Only)")
        print(f"  > Dedicated Low-Volatility ETFs : {low_vol_etfs:>5.1f}%")
        print(f"  > Defensive Individual Stocks   : {defensive_stocks:>5.1f}%")
        print(f"  -----------------------------------------")
        print(f"  > Total Defensive / Low Beta    : {total_defensive_tilt:>5.1f}% of Equity")
        print(f"  > Aggressive Tech / High Beta   : {tech_stocks:>5.1f}% of Equity")

        # Verdict logic
        if total_defensive_tilt > 35:
            print("  [Verdict]: Successfully constructed for LOWER volatility than the S&P 500.")
            print("             The manager is effectively using Min-Vol ETFs alongside defensive stock picking to cushion drawdowns.")
        elif tech_stocks > 35:
            print("  [Verdict]: Constructed for HIGHER volatility than the S&P 500.")
            print("             Heavy tech concentration will cause this portfolio to experience sharper swings than the broader market.")
        else:
            print("  [Verdict]: Neutral Market Weight.")
            print("             This portfolio closely tracks standard broad-market volatility.")

# Execute the combined view and tilt analysis
generate_tilt_and_interactive_view(enriched_df)

In [ ]:
# Force display to show in the cell output
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 1000)

# Check if enriched_df has rows
print(f"Total rows in enriched_df: {len(enriched_df)}")

# Generate and print directly
hierarchical_df = enriched_df.set_index(['Account', 'Asset_Class', 'Sector_Category', 'Symbol'])
print(hierarchical_df[['Asset_Name', 'Market_Value']].to_string())

In [ ]:
# %%
import pandas as pd

excel_output_path = "portfolio_master_with_tilts.xlsx"

with pd.ExcelWriter(excel_output_path, engine='openpyxl') as writer:
    export_df = enriched_df.copy()

    # 1. Add the Macro_Class
    def classify_macro(row):
        cat = str(row.get('Sector_Category', '')).lower()
        name = str(row.get('Asset_Name', '')).lower()
        ac = str(row.get('Asset_Class', '')).lower()

        if 'muni' in cat or 'bond' in cat or 'fixed income' in cat or 'cash' in ac or 'muni' in name or 'treasury' in cat or 'tips' in cat:
            return 'Fixed Income & Cash'
        return 'Equity'

    export_df['Macro_Class'] = export_df.apply(classify_macro, axis=1)

    # 2. Add the specific Tilt_Category for PivotTable drill-downs
    def classify_tilt(row):
        if row['Macro_Class'] == 'Fixed Income & Cash':
            return 'Fixed Income / Cash'

        ac = str(row.get('Asset_Class', ''))
        cat = str(row.get('Sector_Category', ''))

        # Identify Low Volatility ETFs
        if ac == 'ETF / Index' and ('low volatility' in cat.lower() or 'min vol' in cat.lower()):
            return 'Defensive (Low-Vol ETF)'

        # Identify Individual Stock Tilts
        if ac == 'Individual Security':
            if any(def_sec in cat for def_sec in ['Healthcare', 'Utilities', 'Consumer Defensive', 'Consumer Staples']):
                return 'Defensive (Individual Stock)'
            if 'Technology' in cat or 'Communication' in cat:
                return 'Aggressive Tech / High Beta'

        # Everything else (S&P 500 ETFs, Industrials, Financials, etc.)
        return 'Core / Neutral Market'

    export_df['Tilt_Category'] = export_df.apply(classify_tilt, axis=1)

    # 3. Master sheet with all enriched rows and new tags
    # Calculate global weights for the master sheet
    total_portfolio_val = export_df['Market_Value'].sum()
    export_df['%_of_Total_Portfolio'] = (export_df['Market_Value'] / total_portfolio_val) * 100

    master_cols = ['Account', 'Symbol', 'Asset_Name', 'Macro_Class', 'Tilt_Category', 'Asset_Class', 'Sector_Category', '%_of_Total_Portfolio', 'Market_Value']

    export_df[master_cols].to_excel(writer, sheet_name="Master Data (For Pivot)", index=False)

    # 4. Account-specific breakdown tabs
    for acc in export_df['Account'].unique():
        acc_df = export_df[export_df['Account'] == acc].copy()

        total_val = acc_df['Market_Value'].sum()
        acc_df['%_of_Account'] = (acc_df['Market_Value'] / total_val) * 100 if total_val > 0 else 0.0

        acc_df = acc_df.sort_values(
            by=['Macro_Class', 'Tilt_Category', 'Asset_Class', 'Sector_Category', 'Market_Value'],
            ascending=[False, True, True, True, False]
        )

        clean_sheet_name = str(acc)[:28].replace(':', '').replace('/', '').replace('\\', '').replace('?', '').replace('*', '')

        acc_cols = ['Symbol', 'Asset_Name', 'Macro_Class', 'Tilt_Category', 'Asset_Class', 'Sector_Category', '%_of_Account', 'Market_Value']
        acc_df[acc_cols].to_excel(writer, sheet_name=clean_sheet_name, index=False)

print(f"Export complete. Master data with Tilt tags saved to: {excel_output_path}")

In [ ]:
# %%
import pandas as pd

def export_volatility_analysis_to_excel(df, output_path="portfolio_tilt_summary.xlsx"):
    """
    Calculates equity tilt and volatility metrics per account and exports them
    as columns to an Excel file for PivotTable analysis.
    """
    if df is None or df.empty:
        print("No data to process.")
        return

    # Ensure Macro_Class is present
    if 'Macro_Class' not in df.columns:
        def classify_macro(row):
            cat = str(row.get('Sector_Category', '')).lower()
            name = str(row.get('Asset_Name', '')).lower()
            ac = str(row.get('Asset_Class', '')).lower()

            if 'muni' in cat or 'bond' in cat or 'fixed income' in cat or 'cash' in ac or 'muni' in name or 'treasury' in cat or 'tips' in cat:
                return 'Fixed Income & Cash'
            return 'Equity'
        df['Macro_Class'] = df.apply(classify_macro, axis=1)

    defensive_sectors = ['Healthcare', 'Utilities', 'Consumer Defensive', 'Consumer Staples']
    tilt_records = []

    for acc in df['Account'].unique():
        acc_df = df[df['Account'] == acc]

        # Isolate the Equity Portion for precise tilt calculations
        eq_df = acc_df[acc_df['Macro_Class'] == 'Equity'].copy()
        total_eq = eq_df['Market_Value'].sum()

        if total_eq == 0:
            continue

        eq_df['Eq_Weight_%'] = (eq_df['Market_Value'] / total_eq) * 100

        # Calculate specific tilts
        low_vol_etfs = eq_df[(eq_df['Asset_Class'] == 'ETF / Index') &
                              (eq_df['Sector_Category'].str.contains('Low Volatility|Min Vol', case=False, na=False))]['Eq_Weight_%'].sum()

        defensive_stocks = eq_df[(eq_df['Asset_Class'] == 'Individual Security') &
                                  (eq_df['Sector_Category'].str.contains('|'.join(defensive_sectors), case=False, na=False))]['Eq_Weight_%'].sum()

        tech_stocks = eq_df[(eq_df['Asset_Class'] == 'Individual Security') &
                             (eq_df['Sector_Category'].str.contains('Technology|Communication', case=False, na=False))]['Eq_Weight_%'].sum()

        total_defensive_tilt = low_vol_etfs + defensive_stocks

        # Structure the verdict for categorical filtering in Excel
        if total_defensive_tilt > 35:
            verdict = "Lower Volatility (Defensive Tilt)"
        elif tech_stocks > 35:
            verdict = "Higher Volatility (Tech Heavy)"
        else:
            verdict = "Neutral Market Weight"

        # Append as a structured row
        tilt_records.append({
            'Account': acc,
            'Total_Equity_Value': total_eq,
            'Low_Vol_ETFs_%': round(low_vol_etfs, 2),
            'Defensive_Stocks_%': round(defensive_stocks, 2),
            'Total_Defensive_Tilt_%': round(total_defensive_tilt, 2),
            'Aggressive_Tech_%': round(tech_stocks, 2),
            'Volatility_Verdict': verdict
        })

    summary_df = pd.DataFrame(tilt_records)

    # Write to Excel
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        summary_df.to_excel(writer, sheet_name="Account Tilt Summary", index=False)
        df.to_excel(writer, sheet_name="Raw Master Data", index=False)

    print(f"Export complete. File saved to: {output_path}")
    return summary_df

# Execute the export using your enriched dataframe
summary_df = export_volatility_analysis_to_excel(enriched_df)
display(summary_df)